In [ ]:
import os
import json
import torch
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from utils import get_metrics
from dataset import get_dataloaders
from model import BaselineCNN, ResNet50Cassava, EfficientNetV2SCassava, DINOv2Cassava

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os

In [ ]:


def run_cassava_evaluation():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    NUM_CLASSES = 5
    MODELS_DIR = '/content/drive/MyDrive/Cassava_Project/Ֆայլեր'
    MODEL_SAVE_DIR = '/content/drive/MyDrive/Cassava_Project/checkpoints'
    IMAGES_DIR = 'cassava_data/train_images'

    RESULTS_DIR = '/content/drive/MyDrive/Cassava_Project/results'
    os.makedirs(MODELS_DIR, exist_ok=True)
    os.makedirs(RESULTS_DIR, exist_ok=True)


    CNN_PATH = os.path.join(MODEL_SAVE_DIR, 'baseline_cnn_best.pth')
    RESNET_FINETUNED_PATH = os.path.join(MODEL_SAVE_DIR, 'resnet50_finetuned_best.pth')
    EFFNET_PATH = os.path.join(MODEL_SAVE_DIR, 'efficientnet_v2_s_head_best.pth')
    best_dino_path = os.path.join(MODEL_SAVE_DIR, 'dinov2_vitb14_best.pth')

    # 1. Տվյալների բաժանում և դասերի բեռնում
    df = pd.read_csv('cassava_data/train.csv')
    train_df, val_df = train_test_split(df, test_size=0.20, random_state=42, stratify=df['label'])

    with open('cassava_data/label_num_to_disease_map.json') as f:
        label_map = json.load(f)
    class_names = [label_map[str(i)] for i in range(NUM_CLASSES)]

    # 2. DataLoader-ների ստեղծում (ստանդարտ 384px և DINOv2 224px)
    _, valid_loader = get_dataloaders(
        train_df=train_df, val_df=val_df, img_dir=IMAGES_DIR, img_size=384, batch_size=16
    )
    _, dinov2_valid_loader = get_dataloaders(
        train_df=train_df, val_df=val_df, img_dir=IMAGES_DIR, img_size=224, batch_size=16
    )

    # 3. Մոդելների ստեղծում և բեռնում
    baseline_model = BaselineCNN(num_classes=NUM_CLASSES).to(device)
    dino_model = DINOv2Cassava(num_classes=NUM_CLASSES, freeze_backbone=True).to(device)
    resnet_model = ResNet50Cassava(num_classes=NUM_CLASSES, freeze_backbone=True).to(device)
    effnet_model = EfficientNetV2SCassava(num_classes=NUM_CLASSES, freeze_backbone=True).to(device)

    baseline_model.load_state_dict(torch.load(CNN_PATH, map_location=device))

    resnet_state_dict = torch.load(RESNET_FINETUNED_PATH, map_location=device)
    if not any(k.startswith('resnet.') for k in resnet_state_dict.keys()):
        resnet_state_dict = {f"resnet.{k}": v for k, v in resnet_state_dict.items()}
    resnet_model.load_state_dict(resnet_state_dict, strict=False)

    effnet_state_dict = torch.load(EFFNET_PATH, map_location=device)
    if not any(k.startswith('net.') for k in effnet_state_dict.keys()):
        effnet_state_dict = {f"net.{k}": v for k, v in effnet_state_dict.items()}
    effnet_model.load_state_dict(effnet_state_dict, strict=False)

    dino_model.load_state_dict(torch.load(best_dino_path, map_location=device), strict=False)
    print(" Մոդելների քաշերը հաջողությամբ բեռնվեցին:")

    # 4. Մետրիկների հաշվում
    print("\n=== Baseline CNN ===")
    baseline_metrics = get_metrics(baseline_model, valid_loader, device, class_names)

    print("\n=== ResNet50 (fine-tuned) ===")
    resnet_metrics = get_metrics(resnet_model, valid_loader, device, class_names)

    print("\n=== EfficientNetV2-S (head-only) ===")
    effnet_metrics = get_metrics(effnet_model, valid_loader, device, class_names)

    print("\n=== DINOv2Cassava ===")
    dino_metrics = get_metrics(dino_model, dinov2_valid_loader, device, class_names)

    # 5. Ամփոփում և աղյուսակ
    results = {
        "BaselineCNN": baseline_metrics,
        "ResNet50": resnet_metrics,
        "EfficientNetV2S": effnet_metrics,
        "DINOv2Cassava": dino_metrics,
    }

    results_df = pd.DataFrame(results).T
    results_df = results_df[["accuracy", "macro_precision", "macro_recall", "macro_f1"]]

    print("\n=== Final Comparison Summary ===")
    print(results_df.round(5))

    return results_df

In [ ]:
def plot_model_results(results_df, results_dir='/content/drive/MyDrive/Cassava_Project/results'):
    results_df_display = results_df.reset_index()
    results_df_display.rename(columns={'index': 'Model'}, inplace=True)

    colors = ['#888888', '#4C72B0', '#55A868', '#C44E52']
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    metrics_list = [
        ("accuracy", "Accuracy Comparison"),
        ("macro_f1", "Macro F1 Comparison"),
        ("macro_precision", "Macro Precision Comparison"),
        ("macro_recall", "Macro Recall Comparison")
    ]

    for idx, (m_key, m_title) in enumerate(metrics_list):
        row, col = idx // 2, idx % 2
        axes[row][col].bar(
            results_df_display["Model"],
            results_df_display[m_key],
            color=colors[:len(results_df_display)]
        )
        axes[row][col].set_title(m_title)
        axes[row][col].set_ylabel(m_key.replace('_', ' ').capitalize())
        axes[row][col].set_ylim(0, 1)
        axes[row][col].tick_params(axis='x', rotation=15)

    plt.tight_layout()
    plt.savefig('model_comparison_chart.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:

def finalize_best_model(results_df, baseline_model, resnet_model, effnet_model, dino_model,
                        valid_loader, dinov2_valid_loader, class_names, device, num_classes):

    # --- Ընտրում ենք հաղթողին ---
    best_model_name = results_df.sort_values("macro_f1", ascending=False).index[0]
    print(f" Best model by Macro F1: {best_model_name}")

    model_lookup = {
        "BaselineCNN": baseline_model,
        "ResNet50": resnet_model,
        "EfficientNetV2S": effnet_model,
        "DINOv2Cassava": dino_model,
    }
    loader_lookup = {
        "BaselineCNN": valid_loader,
        "ResNet50": valid_loader,
        "EfficientNetV2S": valid_loader,
        "DINOv2Cassava": dinov2_valid_loader,
    }

    final_model = model_lookup[best_model_name]
    final_loader = loader_lookup[best_model_name]
    final_model_name = best_model_name

    # --- Պահպանում ենք վերջնական մոդելը ---
    save_path = '/content/drive/MyDrive/Cassava_Project/final_model.pth'
    torch.save({
        'model_state_dict': final_model.state_dict(),
        'model_name': final_model_name,
        'num_classes': num_classes,
        'class_names': class_names,
    }, save_path)
    print(f" Final model saved: {final_model_name} -> {save_path}")

    # --- Մանրամասն հաշվետվություն ---
    final_model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for inputs, labels in final_loader:
            inputs = inputs.to(device)
            outputs = final_model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(labels.numpy())

    accuracy = accuracy_score(all_targets, all_preds)
    macro_precision = precision_score(all_targets, all_preds, average='macro')
    macro_recall = recall_score(all_targets, all_preds, average='macro')
    macro_f1 = f1_score(all_targets, all_preds, average='macro')
    weighted_f1 = f1_score(all_targets, all_preds, average='weighted')

    per_class_precision = precision_score(all_targets, all_preds, average=None)
    per_class_recall = recall_score(all_targets, all_preds, average=None)
    per_class_f1 = f1_score(all_targets, all_preds, average=None)

    print(f"{'='*60}\nFINAL EVALUATION REPORT: {final_model_name}\n{'='*60}\n")
    print(f"Overall Accuracy:  {accuracy:.4f}")
    print(f"Macro Precision:   {macro_precision:.4f}")
    print(f"Macro Recall:      {macro_recall:.4f}")
    print(f"Macro F1:          {macro_f1:.4f}")
    print(f"Weighted F1:       {weighted_f1:.4f}\n")
    print(classification_report(all_targets, all_preds, target_names=class_names, digits=4))

    per_class_df = pd.DataFrame({
        "Class": class_names,
        "Precision": per_class_precision.round(4),
        "Recall": per_class_recall.round(4),
        "F1": per_class_f1.round(4),
    })
    cm = confusion_matrix(all_targets, all_preds)

    # --- Confusion matrix-ի պահպանում գրաֆիկի տեսքով ---
    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
    plt.title(f"Confusion Matrix — {final_model_name}")
    plt.colorbar()
    tick_marks = np.arange(len(class_names))
    plt.xticks(tick_marks, class_names, rotation=45, ha='right')
    plt.yticks(tick_marks, class_names)
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, format(cm[i, j], "d"), ha="center",
                     color="white" if cm[i, j] > thresh else "black")
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.tight_layout()
    plt.savefig('/content/drive/MyDrive/Cassava_Project/final_model_confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()

    # --- Արդյունքների CSV ֆայլերի գրանցում ---
    per_class_df.to_csv('/content/drive/MyDrive/Cassava_Project/final_evaluation_per_class.csv', index=False)
    pd.DataFrame([{
        "Model": final_model_name,
        "Accuracy": round(accuracy, 4),
        "Macro Precision": round(macro_precision, 4),
        "Macro Recall": round(macro_recall, 4),
        "Macro F1": round(macro_f1, 4),
        "Weighted F1": round(weighted_f1, 4),
    }]).to_csv('/content/drive/MyDrive/Cassava_Project/final_evaluation_summary.csv', index=False)

    print(" Saved final evaluation summary + per-class CSV + confusion matrix")

In [ ]:
def compute_predictions(final_model, final_loader, device):
    final_model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for inputs, labels in final_loader:
            inputs = inputs.to(device)
            outputs = final_model(inputs)
            _, preds = torch.max(outputs, 1)
            y_pred.extend(preds.cpu().numpy())
            y_true.extend(labels.numpy())
    return np.array(y_true), np.array(y_pred)

In [ ]:
def plot_misclassified_images(y_true, y_pred, final_model_name, final_loader, val_df, images_dir, results_dir):
    cassava_classes = ['CBB', 'CBSD', 'CGM', 'CMD', 'Healthy']

    misclassified_idx = np.where(y_true != y_pred)[0]
    print(f"Ընդհանուր սխալ դասակարգված օրինակներ՝ {len(misclassified_idx)} / {len(y_true)} "
          f"({len(misclassified_idx)/len(y_true)*100:.2f}%)")

    if len(misclassified_idx) == 0:
        print("Հիանալի է, սխալներ չկան!")
        return

    np.random.seed(42)
    sample_idx = np.random.choice(misclassified_idx, size=min(10, len(misclassified_idx)), replace=False)

    n_cols = 5
    n_rows = int(np.ceil(len(sample_idx) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 4 * n_rows))
    axes = axes.flatten()

    val_df_reset = val_df.reset_index(drop=True)

    for ax, idx in zip(axes, sample_idx):
        row = val_df_reset.iloc[idx]
        img_path = os.path.join(images_dir, row['image_id'])
        img = Image.open(img_path).convert("RGB")

        true_label = cassava_classes[y_true[idx]]
        pred_label = cassava_classes[y_pred[idx]]

        ax.imshow(img)
        ax.set_title(f"True: {true_label}\nPred: {pred_label}", fontsize=10, color='red')
        ax.axis('off')

    for ax in axes[len(sample_idx):]:
        ax.axis('off')

    plt.suptitle(f"Misclassified Images — {final_model_name}", fontsize=14)
    plt.tight_layout()

    save_path = os.path.join(results_dir, 'misclassified_images.png')
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()
    print(f"Սխալ նկարների գրաֆիկը պահվեց՝ {save_path}")

In [ ]:


def analyze_confusion_errors(eval_results, y_true, y_pred, val_df_reset, images_dir, results_dir):
    cassava_classes = ['CBB', 'CBSD', 'CGM', 'CMD', 'Healthy']
    cm = eval_results["confusion_matrix"]


    confusion_pairs = []
    for i in range(len(cassava_classes)):
        for j in range(len(cassava_classes)):
            if i != j and cm[i, j] > 0:
                confusion_pairs.append({
                    "True Class": cassava_classes[i],
                    "Predicted As": cassava_classes[j],
                    "Count": cm[i, j]
                })

    confusion_pairs_df = pd.DataFrame(confusion_pairs).sort_values("Count", ascending=False)
    print("=== Ամենաշատ շփոթվող class-զույգերը ===")
    print(confusion_pairs_df.head(10).to_string(index=False))

    if len(confusion_pairs_df) == 0:
        print(" Շփոթություններ չկան confusion matrix-ում!")
        return

    #  Ամենավատ զույգը
    top_pair = confusion_pairs_df.iloc[0]
    print(f"\nԱմենամեծ խնդիրը՝ {top_pair['True Class']} → սխալմամբ դասակարգվում է "
          f"որպես {top_pair['Predicted As']} ({top_pair['Count']} անգամ)")

    #  Տեսնենք այդ զույգի իրական նկարները
    worst_true_idx = cassava_classes.index(top_pair["True Class"])
    worst_pred_idx = cassava_classes.index(top_pair["Predicted As"])

    target_indices = np.where((y_true == worst_true_idx) & (y_pred == worst_pred_idx))[0]

    if len(target_indices) > 0:
        sample_worst = np.random.choice(target_indices, size=min(8, len(target_indices)), replace=False)

        fig, axes = plt.subplots(2, 4, figsize=(16, 8))
        axes = axes.flatten()
        for ax, idx in zip(axes, sample_worst):
            row = val_df_reset.iloc[idx]
            img_path = os.path.join(images_dir, row['image_id'])
            img = Image.open(img_path).convert("RGB")
            ax.imshow(img)
            ax.set_title(f"{top_pair['True Class']} → {top_pair['Predicted As']}", fontsize=10, color='darkred')
            ax.axis('off')

        for ax in axes[len(sample_worst):]:
            ax.axis('off')

        plt.suptitle(f"Worst Confusion: {top_pair['True Class']} misclassified as {top_pair['Predicted As']}", fontsize=14)
        plt.tight_layout()

        save_img_path = os.path.join(results_dir, 'error_analysis_worst_pair.png')
        plt.savefig(save_img_path, dpi=200, bbox_inches='tight')
        plt.show()

    #  Պահպանում ենք ամբողջական error-pair աղյուսակը Drive-ում
    save_csv_path = os.path.join(results_dir, 'error_analysis_confused_pairs.csv')
    confusion_pairs_df.to_csv(save_csv_path, index=False)
    print(f"\n Saved: {save_csv_path}, {save_img_path}")

In [3]:
def show_gradcam_sample(row, model, transform, device, class_names, results_dir):
    img_path = os.path.join(IMAGES_DIR, row['image_id'])
    orig_img = Image.open(img_path).convert("RGB")
    img_tensor = transform(orig_img)

    importance_map = get_dino_attention_map(model, img_tensor, device)
    importance_map_resized = np.array(Image.fromarray(importance_map).resize((224, 224), Image.BILINEAR))

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(orig_img.resize((224, 224)))
    axes[0].set_title(f"Original ({class_names[row['label']]})")
    axes[0].axis('off')

    axes[1].imshow(orig_img.resize((224, 224)))
    axes[1].imshow(importance_map_resized, cmap='jet', alpha=0.5)
    axes[1].set_title("DINOv2 Attention Map")
    axes[1].axis('off')

    plt.tight_layout()

    # Պահում ենք Drive-ում
    save_path = os.path.join(results_dir, f"dino_attention_class_{row['label']}.png")
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import os
import numpy as np

def plot_random_validation_samples(y_true, y_pred, val_df, images_dir, results_dir):
    cassava_classes = ['CBB', 'CBSD', 'CGM', 'CMD', 'Healthy']
    val_df_reset = val_df.reset_index(drop=True)

    np.random.seed(7)
    sample_idx = np.random.choice(len(val_df_reset), size=8, replace=False)

    fig, axes = plt.subplots(2, 4, figsize=(18, 9))
    axes = axes.flatten()

    for ax, idx in zip(axes, sample_idx):
        row = val_df_reset.iloc[idx]
        img_path = os.path.join(images_dir, row['image_id'])
        img = Image.open(img_path).convert("RGB")

        true_label = cassava_classes[y_true[idx]]
        pred_label = cassava_classes[y_pred[idx]]
        color = 'green' if true_label == pred_label else 'red'

        ax.imshow(img)
        ax.set_title(f"True: {true_label}\nPred: {pred_label}", fontsize=11, color=color)
        ax.axis('off')

    plt.suptitle("Random Validation Samples — True vs Predicted", fontsize=14)
    plt.tight_layout()

    plots_dir = os.path.join(results_dir, 'plots')
    os.makedirs(plots_dir, exist_ok=True)

    save_path = os.path.join(plots_dir, 'random_val_predictions.png')
    plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show()
    print(f" Պատահական նմուշների նկարը պահվեց՝ {save_path}")